### 🏷️ **Credits & License**

* 🔗 [OmniVoice GitHub Repository](https://github.com/k2-fsa/OmniVoice)
* 🤗 [OmniVoice on Hugging Face](https://huggingface.co/k2-fsa/OmniVoice)
* 📄 **License**: Provided under the [Apache License 2.0](https://github.com/k2-fsa/OmniVoice/blob/master/LICENSE)



### ⚠️ **Usage Disclaimer**

Use of this voice cloning model is subject to strict ethical and legal standards. By using this tool, you agree **not to** engage in any of the following prohibited activities:

* **Fraud or Deception**: Using cloned voices to create misleading or fraudulent content.
* **Impersonation**: Replicating someone's voice without their explicit permission, especially for malicious, harmful, or deceptive purposes.
* **Illegal Activities**: Employing the model in any manner that violates local, national, or international laws and regulations.
* **Harmful Content Generation**: Creating offensive, defamatory, or unethical material, including content that spreads misinformation or causes harm.

> ⚖️ **Legal Responsibility**
> The developers of this tool disclaim all liability for misuse. **Users bear full responsibility** for ensuring that their usage complies with all applicable laws, regulations, and ethical guidelines.


In [ ]:
#@title Install OmniVoice
%cd /content/
!rm -rf ./omnivoice-colab
!git clone https://github.com/hirannalaka19/omnivoice-colab.git
%cd ./omnivoice-colab
# OmniVoice is now an official pip package: https://github.com/k2-fsa/OmniVoice
!pip install -r colab.txt
from IPython.display import clear_output
clear_output()
print("✅ Installation complete")

In [ ]:
#@title Run Gradio APP
#@markdown **HF_TOKEN** (required): paste a free Hugging Face token — create one at https://huggingface.co/settings/tokens (type: **Read**). It unlocks full download speed.
HF_TOKEN = "" #@param {type:"string"}

import os, subprocess, sys

if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    print("✅ HF token set")
else:
    print("⚠️ No HF_TOKEN provided — Hugging Face throttles or blocks anonymous downloads. Paste a token above for full speed.")

# Hugging Face has two download paths for this model and both can misbehave
# on Colab: the plain HTTP path is fast but sometimes hits CDN signature
# errors (403 "invalid key pair id"), while the Xet protocol path is reliable
# but sometimes slow. Try fast HTTP first, fall back to Xet automatically.
def download_model(disable_xet):
    env = dict(os.environ)
    if disable_xet:
        env["HF_HUB_DISABLE_XET"] = "1"
    else:
        env.pop("HF_HUB_DISABLE_XET", None)
    return subprocess.call(
        [sys.executable, "-c",
         "from huggingface_hub import snapshot_download; snapshot_download('k2-fsa/OmniVoice')"],
        env=env,
    )

print("⬇️ Downloading the model (fast HTTP path) ...")
if download_model(disable_xet=True) != 0:
    print("⚠️ Fast HTTP path failed (Hugging Face CDN issue) — retrying with the Xet protocol (can be slower) ...")
    if download_model(disable_xet=False) != 0:
        raise SystemExit("❌ Both download paths failed — re-run this cell to retry.")
print("✅ Model downloaded")

# Model is now cached locally, so the app starts without re-downloading.
os.environ["HF_HUB_DISABLE_XET"] = "1"

%cd /content/omnivoice-colab
!python app.py